In [0]:
from pyspark.sql import functions as F

df = spark.table("workspace.pobreza.ech_2024_bronze")

# 1) Todas las columnas, en una sola línea para copiar fácil
print(len(df.columns), "columnas")
print(", ".join(df.columns))

# 2) Valores de las columnas candidatas a educación y empleo
import re
candidatas = [c for c in df.columns
              if re.search(r"(?i)pobpcoac|asist|niv|educ|anio|^e19|^e2[0-4]|^e4[89]|^e5[0-9]|^f6[6-9]|^f7", c)]
print("\nCandidatas:", candidatas)
for c in candidatas[:25]:
    vals = df.groupBy(c).count().orderBy(F.desc("count")).limit(8).collect()
    print(f"\n{c}: " + " | ".join(f"{r[0]}→{r[1]}" for r in vals))

In [0]:
from pyspark.sql import functions as F

df = spark.table("workspace.pobreza.ech_2024_bronze")
w = F.col("W_ANO")
def pct(cond):  # % ponderado
    return F.round(F.sum(F.when(cond, w).otherwise(0)) / F.sum(w) * 100, 1)

# A) ¿e49 = 3 es "asiste ahora"? Si sí, casi todos los de 6-14 tienen 3
display(df.withColumn("tramo", F.when(F.col("e27") < 6, "0-5").when(F.col("e27") < 15, "6-14")
                               .when(F.col("e27") < 25, "15-24").otherwise("25+"))
          .groupBy("tramo").pivot("e49").count().orderBy("tramo"))

# B) Valores de las variables de finalización y terciaria
for c in ["e51_5", "e201_1c", "e51_6", "e201_1d", "e209_1", "e51_8", "e51_9", "e51_10", "e51_11"]:
    print(c, df.groupBy(c).count().orderBy(F.desc("count")).limit(6).collect())

# C) Chequeos contra datos oficiales
termino_media = ((F.col("e201_1c") == 1) | (F.col("e201_1d") == 1) |
                 (F.col("e51_8") > 0) | (F.col("e51_9") > 0) | (F.col("e51_10") > 0) | (F.col("e51_11") > 0))
activo = F.col("POBPCOAC").between(2, 5)
display(df.agg(
    pct(termino_media & F.col("e27").between(21, 23)).alias("_"),
).limit(0))  # placeholder para que display no falle

print("Egreso media 21-23 (INEEd ≈ 52%):",
      df.filter(F.col("e27").between(21, 23)).agg(pct(termino_media)).first()[0])
print("Desempleo 14-24 (OIT ≈ 25%):",
      df.filter(F.col("e27").between(14, 24) & activo).agg(pct(F.col("POBPCOAC").between(3, 5))).first()[0])
print("Ni estudian ni trabajan 15-24:",
      df.filter(F.col("e27").between(15, 24)).agg(pct((F.col("e49") != 3) & (F.col("POBPCOAC") != 2))).first()[0])
print("POBPCOAC:", df.groupBy("POBPCOAC").count().orderBy("POBPCOAC").collect())

In [0]:
from pyspark.sql import functions as F

NOMBRES = {1: "Montevideo", 2: "Artigas", 3: "Canelones", 4: "Cerro Largo", 5: "Colonia", 6: "Durazno",
           7: "Flores", 8: "Florida", 9: "Lavalleja", 10: "Maldonado", 11: "Paysandú", 12: "Río Negro",
           13: "Rivera", 14: "Rocha", 15: "Salto", 16: "San José", 17: "Soriano", 18: "Tacuarembó", 19: "Treinta y Tres"}
mapa = F.create_map([F.lit(x) for kv in NOMBRES.items() for x in kv])

termino_media = ((F.col("e201_1c") == 1) | (F.col("e201_1d") == 1) | (F.col("e51_8") > 0) |
                 (F.col("e51_9") > 0) | (F.col("e51_10") > 0) | (F.col("e51_11") > 0))

educ = (spark.table("workspace.pobreza.ech_2024_bronze").select(
        F.col("e27").cast("int").alias("edad"),
        F.when(F.col("e26") == 1, "Hombres").otherwise("Mujeres").alias("sexo"),
        F.col("dpto").cast("int").alias("dpto"),
        mapa[F.col("dpto")].alias("departamento"),
        termino_media.alias("termino_media"),
        (F.col("e49") == 3).alias("estudia"),
        (F.col("POBPCOAC") == 2).alias("ocupado"),
        F.col("POBPCOAC").between(3, 5).alias("desocupado"),
        F.col("W_ANO").alias("peso")))

educ.write.mode("overwrite").option("overwriteSchema", True).saveAsTable("workspace.pobreza.educacion_silver")
display(educ.limit(5))

In [0]:
import os

def carpeta_data():
    ruta = os.path.dirname(dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get())
    if not ruta.startswith("/Workspace"):
        ruta = "/Workspace" + ruta
    if os.path.basename(ruta) == "notebooks":   # si estoy en la carpeta notebooks...
        ruta = os.path.dirname(ruta)            # ...subo un nivel, a la raíz del repo
    os.makedirs(f"{ruta}/data", exist_ok=True)
    return f"{ruta}/data"

def guardar_gold(df, nombre):
    df.write.mode("overwrite").option("overwriteSchema", True).saveAsTable(f"workspace.pobreza.{nombre}_gold")
    df.toPandas().to_json(f"{carpeta_data()}/{nombre}.json", orient="records", force_ascii=False, indent=1)
    print("OK:", nombre)

In [0]:
e = spark.table("workspace.pobreza.educacion_silver")
p = lambda cond: F.round(F.sum(F.when(cond, F.col("peso")).otherwise(0)) / F.sum("peso") * 100, 1)
n = F.count("*").alias("n_muestra")

# 1) Egreso de media superior por generación y sexo
gen = (F.when(F.col("edad").between(21, 23), "21-23").when(F.col("edad").between(25, 29), "25-29")
        .when(F.col("edad").between(30, 39), "30-39").when(F.col("edad").between(40, 49), "40-49")
        .when(F.col("edad").between(50, 64), "50-64").when(F.col("edad") >= 65, "65+"))
egreso = (e.withColumn("generacion", gen).filter(F.col("generacion").isNotNull())
          .groupBy("generacion").agg(p(F.col("termino_media")).alias("pct_total"),
              p(F.col("termino_media") & (F.col("sexo") == "Hombres")) .alias("_h"), n)
          .drop("_h"))
egreso_sexo = (e.withColumn("generacion", gen).filter(F.col("generacion").isNotNull())
               .groupBy("generacion", "sexo").agg(p(F.col("termino_media")).alias("pct")))
egreso = egreso.join(egreso_sexo.groupBy("generacion").pivot("sexo").agg(F.first("pct")), "generacion").orderBy("generacion")

# 2) Ni estudian ni trabajan, 15-24
j = e.filter(F.col("edad").between(15, 24))
nini = ~F.col("estudia") & ~F.col("ocupado")
nini_sexo = j.groupBy("sexo").agg(p(nini).alias("pct_nini"), n).orderBy("sexo")
nini_dpto = j.groupBy("dpto", "departamento").agg(p(nini).alias("pct_nini"), n).orderBy("dpto")

# 3) Desempleo de 18-29 según si terminó el liceo
act = e.filter(F.col("edad").between(18, 29) & (F.col("ocupado") | F.col("desocupado")))
desempleo_educ = (act.withColumn("educacion", F.when(F.col("termino_media"), "Terminó el liceo").otherwise("No terminó el liceo"))
                  .groupBy("educacion").agg(p(F.col("desocupado")).alias("pct_desempleo"), n))

guardar_gold(egreso, "educacion_egreso_generacion")
guardar_gold(nini_sexo, "educacion_nini_sexo")
guardar_gold(nini_dpto, "educacion_nini_dpto")
guardar_gold(desempleo_educ, "educacion_desempleo")

display(egreso); display(nini_sexo); display(desempleo_educ)